# 📚 03: Semantic Book Embeddings — Goal 1 Foundation (Recommendation Type: Content-Based Filtering)

## 🎯 Objective: Create the Semantic Map of the Book Catalog

This notebook transforms the cleaned book metadata into **384-dimensional semantic vectors (embeddings)** using a pre-trained Sentence-BERT model. These embeddings enable the system to understand conceptual similarity between books — the foundation for **Goal 1: Item-to-Item Semantic Recommendations**.

### What This Enables

| Capability | How This Notebook Delivers |
|------------|---------------------------|
| **Semantic search** | Find books conceptually similar to a query or example book |
| **"You might also like"** | Identify books with similar descriptions, authors, or themes |
| **Content-based filtering** | Generate recommendations without requiring user ratings |
| **Hybrid recommendations (Goal 3)** | Provide semantic similarity scores to blend with collaborative filtering |

---

## 📊 Notebook Overview

| Stage | Description | Key Output |
|-------|-------------|-------------|
| **1. Load Data** | Import cleaned metadata from EDA notebook | `books_data_embeddings` (212,399 books × 9 columns) |
| **2. Generate Embeddings** | Encode `content_for_embedding` using Sentence-BERT | `embeddings_matrix` (212,399 × 384) |
| **3. Save Artifacts** | Store embeddings and metadata separately for fast loading | `.npy` + `.parquet` files |
| **4. Similarity Search** | Demonstrate cosine similarity for book recommendations | Top-N similar books |

---

## 🔧 Step-by-Step Breakdown

| Cell | Task | Key Operation | Why It Matters |
|------|------|---------------|----------------|
| **1** | Environment setup | Import transformers, suppress warnings | Clean execution |
| **2** | Load metadata | `pd.read_parquet()` | 212,399 books loaded in <1 sec |
| **3** | Generate embeddings | `SentenceTransformer.encode()` with batch_size=64 | ~27 min processing time (vs days with pipeline) |
| **4** | Save artifacts | `np.save()` + `to_parquet()` | 311 MB embedding matrix + 295 MB metadata |
| **5** | Verify loading | `np.load()` + `pd.read_parquet()` | Embeddings shape: (212,399, 384) |
| **6-11** | Explore data | Inspect metadata and embedding vectors | Understand data structure |
| **12-14** | Similarity demo | `cosine_similarity()` for "The Science of Cold Case Files" | Top matches: forensic science books (0.66 similarity) |
| **15-17** | Create function | `get_similar_books()` reusable function | Encapsulates similarity search logic |
| **18** | Test function | Recommend books for "The Universe" | Top matches: astronomy/physics books (0.68 similarity) |

---

## 🧠 The Embedding Pipeline

```text
Raw Text (content_for_embedding)
              ↓
"the alchemist paulo coelho fiction my heart is afraid..."
              ↓
[Tokenization] → [101, 2345, 1234, 8923, 102, ...]
              ↓
[Transformer Layers] → 6 layers of self-attention
              ↓
[Mean Pooling] → Average all token vectors
              ↓
[384-dimensional vector] → [0.12, -0.34, 0.56, ..., 0.78]
              ↓
Saved to: books_embeddings_matrix.npy (311 MB)

In [1]:
%%time
# ============================================================
# CELL 1 - Import Transformers and set up the environment
# ============================================================

# add a timer and hide all non-critical warnings
from transformers import pipeline, logging

logging.set_verbosity_error()

CPU times: user 2.36 s, sys: 691 ms, total: 3.05 s
Wall time: 4.11 s


In [2]:
# ============================================================
# CELL 2 - Load books_data_ready_for_embedding.parquet
# ============================================================
import pandas as pd

# Load the optimized checkpoint
books_data_embeddings = pd.read_parquet("../Data/Books_data/books_data_ready_for_embedding.parquet")

print(f"   Loaded successfully")
print(f"   Shape   : {books_data_embeddings.shape}")
print(f"   Columns : {books_data_embeddings.columns.tolist()}")
print(f"   Memory  : {books_data_embeddings.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Quick verification that lists are intact (Checking parsed correctly)
print(f"   Check Type: {type(books_data_embeddings['authors_parsed'].iloc[0])}") 

books_data_embeddings.head(3)

   Loaded successfully
   Shape   : (209452, 9)
   Columns : ['Title', 'description', 'authors_parsed', 'categories_parsed', 'year', 'image', 'infoLink', 'ratingsCount', 'content_for_embedding']
   Memory  : 291.6 MB
   Check Type: <class 'numpy.ndarray'>


,Title,description,authors_parsed,categories_parsed,year,image,infoLink,ratingsCount,content_for_embedding
0,The Alchemist,"""My heart is afraid that it will have to suffe...",[Paulo Coelho],[Fiction],2006.0,http://books.google.com/books/content?id=pTr44...,http://books.google.com/books?id=pTr44Sx6oWQC&...,4895.0,"the alchemist paulo coelho fiction ""my heart i..."
1,A Mission to Millions; The Story of Ernie Alle...,#1 NEW YORK TIMES BESTSELLER • NOW A MAJOR MOT...,[Laura Hillenbrand],[Biography & Autobiography],2014.0,http://books.google.com/books/content?id=1PeLD...,http://books.google.com/books?id=1PeLDQAAQBAJ&...,4572.0,a mission to millions; the story of ernie alle...
2,Unbroken,#1 NEW YORK TIMES BESTSELLER • NOW A MAJOR MOT...,[Laura Hillenbrand],[Biography & Autobiography],2014.0,http://books.google.com/books/content?id=1PeLD...,http://books.google.com/books?id=1PeLDQAAQBAJ&...,4572.0,unbroken laura hillenbrand biography & autobio...


In [ ]:
### DO NOT RUN - IT IS JUST FOR COMPARISON PURPOSES
### Initial Embeddings Creation

### extract the embedding representation for each review
#feature_extractor = pipeline("feature-extraction",
                             #model="sentence-transformers/all-MiniLM-L6-v2",
                             #device='mps',
                             #show_progress_bar=True,
                            #truncation=True,      # ← truncate sequences exceeding 512 tokens
                            #max_length=512        # ← explicitly set the max token limit))

#embeddings = books_data_embeddings['description'].apply(lambda x: feature_extractor(x)[0][0])
#embeddings.head(2)

### CELL 3: Generating Book Embeddings for Semantic Search

#### What This Code Does

This cell transforms the cleaned book metadata into **numerical vectors (embeddings)** that capture the semantic meaning of each book. These embeddings enable the system to find conceptually similar books — for example, recognizing that a book about "WWII fighter pilots" belongs near "aviation history memoirs" even when the exact words don't match.

---

#### Technical Breakdown

| Step | Action | Purpose |
|------|--------|---------|
| **1** | Load `all-MiniLM-L6-v2` onto Apple MPS (GPU) | Accelerate inference 10-20x vs CPU |
| **2** | Extract `content_for_embedding` column | Combines Title + authors + categories + description into one semantic string |
| **3** | Encode in batches of 64 books | GPU processes multiple texts in parallel for efficiency |
| **4** | Apply **mean pooling** to transformer outputs | Converts variable-length text to fixed 384-dimensional vectors |
| **5** | Save as `.npy` file (float32) | Stores ~322 MB embedding matrix for fast loading in downstream notebooks |

---

#### Why This Approach (Not `pipeline()`)

| Issue with `pipeline()` | Solution with `SentenceTransformer` |
|------------------------|-------------------------------------|
| Returns nested `[batch][seq][features]` structure | Direct `.encode()` returns clean numpy array |
| No native batching → 12+ days processing | Batch size 64 → ~10 minutes |
| Stores embeddings in DataFrame → RAM explosion | Saves directly to `.npy` → memory efficient |
| Sequential processing on CPU | Parallel processing on MPS GPU |

---

#### The Embedding Pipeline Visualized

| Step | Process | Output |
|------|---------|--------|
| 1 | Raw text input | `"the alchemist paulo coelho fiction..."` |
| 2 | Tokenization | `[101, 2345, 1234, 8923, 102, ...]` |
| 3 | Transformer layers | 6 layers of self-attention |
| 4 | Mean pooling | Average of all token vectors |
| 5 | Final embedding | `[0.12, -0.34, 0.56, ..., 0.78]` (384-dim) |
| 6 | Persistence | `books_embeddings_matrix.npy` (322 MB) |

"feature-extraction provides everything, even the tokens — which makes the code collapse due to the massive amount of nested data structures."

*"SentenceTransformer goes straight to the final embedding (CLS or mean pooled) and discards the tokens, returning only the 384-dimensional vector."*

NOTE: Pre-trained Sentence-BERT converts each book description into a 384-dimensional vector based on semantic patterns learned from millions of text pairs. Cosine similarity then measures the angular distance between these vectors — books with similar meanings cluster together, enabling 'if you liked X, you'll love Y' recommendations.

In [ ]:
# ============================================================ 
# CELL 3 - Extract embedding representations for each description
# ============================================================
from sentence_transformers import SentenceTransformer
import torch
import numpy as np

# 1. Load the pre-trained SentenceTransformer model
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='mps') # Class constructs the model and moves it to the specified device (MPS for Apple Silicon GPUs)

# 2. Use the built-in .encode method 
# Applies mean/CLS pooling to get fixed-size vector (384-dim)
# CLS (Classification Token) pooling = A student who summarizes an entire book in ONE sentence. Fast, but might miss nuance.
print(f"Encoding {len(books_data_embeddings):,} books...")
embeddings_matrix = model.encode(
    books_data_embeddings['content_for_embedding'].tolist(), # Model expects a list of strings, not a pandas Series
    batch_size=64, # 64 texts at a time → 64 embeddings computed in parallel on GPU
    show_progress_bar=True,
    convert_to_numpy=True # Directly get a NumPy array instead of a list of lists
)

# 3. Save directly to NumPy (skipping the step of adding it to the DataFrame to save RAM)
np.save("../Data/Books_data/books_embeddings_matrix.npy", embeddings_matrix.astype(np.float32))
print("Saved to .npy successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 209,452 books...


Batches:   0%|          | 0/3273 [00:00<?, ?it/s]

Saved to .npy successfully!


In [ ]:
# ============================================================
# CELL 4 - Persistent Storage for Metadata
# ============================================================
import os

# 1. Save the Metadata Parquet
# Since we didn't add the embeddings to this DataFrame, 
# 'books_data_embeddings' IS already the light metadata-only version.
metadata_path = "../Data/Books_data/books_metadata_only.parquet"
books_data_embeddings.to_parquet(metadata_path, index=False)

# 2. Final Verification
print(f" Metadata saved: {metadata_path}")
print(f" Total books: {len(books_data_embeddings):,}")
print(f" Embedding Matrix is already on disk: ../Data/Books_data/books_embeddings_matrix.npy")

# Quick check on matrix size for your records
matrix_size_mb = os.path.getsize("../Data/Books_data/books_embeddings_matrix.npy") / 1024**2
print(f"  Matrix File Size: {matrix_size_mb:.1f} MB")

 Metadata saved: ../Data/Books_data/books_metadata_only.parquet
 Total books: 209,452
 Embedding Matrix is already on disk: ../Data/Books_data/books_embeddings_matrix.npy
  Matrix File Size: 306.8 MB


In [3]:
# ============================================================
# CELL 5 - Load Metadata and Embeddings for Similarity Search
# ============================================================

import numpy as np
import pandas as pd

# Load in < 1 second
metadata = pd.read_parquet("../Data/Books_data/books_metadata_only.parquet")
vectors = np.load("../Data/Books_data/books_embeddings_matrix.npy")

print(f"Metadata shape: {metadata.shape}")
print(f"Embeddings shape: {vectors.shape}")

Metadata shape: (209452, 9)
Embeddings shape: (209452, 384)


In [4]:
# ============================================================
# CELL 6 - Checking the first few rows of metadata
# ============================================================

metadata.head(2)

,Title,description,authors_parsed,categories_parsed,year,image,infoLink,ratingsCount,content_for_embedding
0,The Alchemist,"""My heart is afraid that it will have to suffe...",[Paulo Coelho],[Fiction],2006.0,http://books.google.com/books/content?id=pTr44...,http://books.google.com/books?id=pTr44Sx6oWQC&...,4895.0,"the alchemist paulo coelho fiction ""my heart i..."
1,A Mission to Millions; The Story of Ernie Alle...,#1 NEW YORK TIMES BESTSELLER • NOW A MAJOR MOT...,[Laura Hillenbrand],[Biography & Autobiography],2014.0,http://books.google.com/books/content?id=1PeLD...,http://books.google.com/books?id=1PeLDQAAQBAJ&...,4572.0,a mission to millions; the story of ernie alle...


In [5]:
# ============================================================
# CELL 7 - Checking the first embedding vector from the loaded .npy file (Books_Embeddings)
# ============================================================

# Quick check on the first embedding vector
vectors[:1]

array([[-1.32764608e-03,  9.65398364e-03,  4.03989255e-02,
         8.99528489e-02, -5.07095791e-02, -2.07398441e-02,
        -3.82493883e-02, -1.87848303e-02,  2.38286536e-02,
        -5.02375886e-02, -4.04486898e-03, -8.00675526e-02,
        -2.06992850e-02,  2.14526616e-02, -8.54798332e-02,
         3.01074497e-02,  1.50437998e-02, -4.68141250e-02,
         5.89134078e-03,  1.32079404e-02,  1.50364721e-02,
         6.67499704e-03, -2.57642586e-02,  6.31241053e-02,
        -2.90627014e-02,  1.28074531e-02,  5.40055288e-03,
        -4.46268469e-02, -6.01670481e-02, -1.46198329e-02,
        -1.27894497e-02,  8.81545153e-03,  2.56056357e-02,
        -2.96181310e-02, -7.76305869e-02, -7.65360193e-03,
         5.17134778e-02,  4.95916568e-02,  2.60413475e-02,
         1.82091463e-02, -2.37192940e-02,  3.37933563e-02,
         4.93680651e-04,  9.01771933e-02,  1.50542939e-02,
        -1.00213565e-01,  1.71933603e-02, -2.14299690e-02,
         2.92827934e-02,  8.95163342e-02, -6.81652501e-0

In [6]:
# ============================================================
# CELL 8 - Search for books related to "Science" in the title
# ============================================================

metadata[metadata['Title'].str.contains('Science', case=False)]

,Title,description,authors_parsed,categories_parsed,year,image,infoLink,ratingsCount,content_for_embedding
319,Gene Hunter: The Story of Neuropsychologist Na...,The #1 NEW YORK TIMES Bestseller The basis for...,[Siddhartha Mukherjee],[Medical],2016.0,http://books.google.com/books/content?id=fOvaC...,https://play.google.com/store/books/details?id...,351.0,gene hunter: the story of neuropsychologist na...
685,Stranger in a Strange Land : ( Science Fiction...,The original uncut edition of STRANGER IN A ST...,[Robert A. Heinlein],[Fiction],2014.0,http://books.google.com/books/content?id=gnwET...,http://books.google.com/books?id=gnwETwF8Zb4C&...,116.0,stranger in a strange land : ( science fiction...
958,Sixth Extinction (Science Masters S.),ONE OF THE NEW YORK TIMES BOOK REVIEW'S 10 BES...,[Elizabeth Kolbert],[Nature],2014.0,http://books.google.com/books/content?id=Ra9RA...,https://play.google.com/store/books/details?id...,78.0,sixth extinction (science masters s.) elizabet...
989,Five Star Science Fiction/Fantasy - Windowpane,"From the author of Exhalation, an award-winnin...",[Ted Chiang],[Fiction],2010.0,http://books.google.com/books/content?id=WTBA-...,https://play.google.com/store/books/details?id...,77.0,five star science fiction/fantasy - windowpane...
1138,Ubik (The Gregg Press science fiction series),Philip K. Dick's searing metaphysical comedy o...,[Philip K. Dick],[Businessmen],1979.0,NaN,http://books.google.com/books?id=6oYvPwAACAAJ&...,65.0,ubik (the gregg press science fiction series) ...
...,...,...,...,...,...,...,...,...,...
208324,Game of Science,Game of Science,[],[],NaN,NaN,NaN,NaN,game of science game of science
208668,"Time Machines: The Story of the Science, Ficti...","Time Machines: The Story of the Science, Ficti...",[Michael Ashley],[Science fiction],2000.0,NaN,http://books.google.com/books?id=_sR_zQEACAAJ&...,NaN,"time machines: the story of the science, ficti..."
208808,Random Field Models in Earth Sciences (Dover B...,Random Field Models in Earth Sciences (Dover B...,[],[],NaN,NaN,NaN,NaN,random field models in earth sciences (dover b...
209396,Science Fiction Fandom,"Many prominent science fiction writers, artist...",[Joseph L. Sanders],[Literary Criticism],1994.0,http://books.google.com/books/content?id=Nr8bA...,http://books.google.com/books?id=Nr8bAQAAIAAJ&...,NaN,science fiction fandom joseph l. sanders liter...


In [7]:
# ============================================================
# CELL 9 - Select the embedding for "Gene Hunter: The Story of Neuropsychologist" (index 319) and view it
# ============================================================

vectors[319]

array([-4.25818488e-02,  3.14338952e-02, -1.24705266e-02,  1.32467952e-02,
       -6.11630119e-02,  1.04904816e-01, -2.35790052e-02,  6.40102103e-02,
       -6.79165684e-03,  2.98476126e-02, -9.43755805e-02, -1.37147540e-02,
       -2.69605126e-02, -2.04984527e-02, -1.05976433e-01,  4.23813798e-02,
       -3.52743790e-02,  4.50201817e-02,  4.69500646e-02, -1.86871570e-02,
       -3.59715745e-02,  5.48707657e-02,  9.02727768e-02, -1.00948848e-01,
        9.81262419e-03,  3.07891462e-02,  2.67440733e-02, -1.03968956e-01,
       -2.88227526e-03,  4.39408459e-02,  1.38724549e-02,  2.50278451e-02,
       -1.08889379e-01,  1.68786962e-02, -5.51471859e-02, -2.89733172e-03,
       -6.23361580e-02,  7.53970519e-02,  5.75516559e-03,  2.39440165e-02,
        8.15092176e-02,  1.30421966e-02, -7.56099001e-02,  5.32779619e-02,
       -8.82497244e-03, -6.35669157e-02, -3.41607109e-02, -1.10649079e-01,
        7.07092229e-03,  2.14631949e-02, -1.16392873e-01, -3.98946144e-02,
        6.27713054e-02, -

In [8]:
print(f"Embedding vector shape is: {vectors[319].shape}")

Embedding vector shape is: (384,)


(384,) ----- One book, 384 features (1D array) ----- [0.12, -0.34, 0.56, ...] (one row)

In [9]:
# ============================================================
# CELL 10 - Save the embedding for that book as a 2D array (row vector)
# ============================================================

import numpy as np

# 2D Array (Row Vector): [[0.1, 0.5, ...]]
# Why reshape(1, -1)? Because many libraries (like scikit-learn) expect a 2D array for distance calculations, even if it's just one vector. This way, we maintain consistency in the data format for future operations.
embedding_science = np.array(vectors[319]).reshape(1, -1)
embedding_science.shape

(1, 384)

(1, 384)-----One book, 384 features (2D array)-----[[0.12, -0.34, 0.56, ...]] (one row in a matrix)

In [10]:
# ============================================================
# CELL 11 - Save the embedding for all books
# ============================================================

# What vstack does is it takes a list of 1D arrays (embeddings) and stacks them vertically to create a 2D array where each row corresponds to a book's embedding. This is the format we need for similarity calculations.
embeddings_books = np.vstack(vectors)
embeddings_books.shape

(209452, 384)

In [11]:
# ============================================================
# CELL 12 - Calculate cosine similarity scores between the selected book and all others
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

similarity_scores_science = cosine_similarity(embedding_science, embeddings_books)
similarity_scores_science_series = pd.Series(similarity_scores_science.flatten(), name='similarity_score')
similarity_scores_science_series.head()

0    0.196022
1    0.232723
2    0.207178
3    0.222041
4    0.226293
Name: similarity_score, dtype: float32

In [12]:
# ============================================================
# CELL 13 - Combine book information with similarity scores
# ============================================================

# combine book titles, descriptions and scores
similarity_scores_science_df = pd.concat([metadata[['Title', 'description']], similarity_scores_science_series], axis=1)
similarity_scores_science_df.head()

,Title,description,similarity_score
0,The Alchemist,"""My heart is afraid that it will have to suffe...",0.196022
1,A Mission to Millions; The Story of Ernie Alle...,#1 NEW YORK TIMES BESTSELLER • NOW A MAJOR MOT...,0.232723
2,Unbroken,#1 NEW YORK TIMES BESTSELLER • NOW A MAJOR MOT...,0.207178
3,Blue Like Jazz: Nonreligious Thoughts on Chris...,A popular minister recounts his zealous early ...,0.222041
4,Eclipse,As the love triangle heats up in the third boo...,0.226293


In [13]:
# ============================================================
# CELL 14 - View the top 5 most similar books to "The Science of Cold Case Files"
# ============================================================

# modify the column width
pd.set_option('display.max_colwidth', None)

# view the top 5 most similar books
similarity_scores_science_df.sort_values('similarity_score', ascending=False).head()

,Title,description,similarity_score
319,Gene Hunter: The Story of Neuropsychologist Nancy Wexler (Women's Adventures in Science (Children's Press)),"The #1 NEW YORK TIMES Bestseller The basis for the PBS Ken Burns Documentary The Gene: An Intimate History Now includes an excerpt from Siddhartha Mukherjee’s new book Song of the Cell! From the Pulitzer Prize–winning author of The Emperor of All Maladies—a fascinating history of the gene and “a magisterial account of how human minds have laboriously, ingeniously picked apart what makes us tick” (Elle). “Sid Mukherjee has the uncanny ability to bring together science, history, and the future in a way that is understandable and riveting, guiding us through both time and the mystery of life itself.” —Ken Burns “Dr. Siddhartha Mukherjee dazzled readers with his Pulitzer Prize-winning The Emperor of All Maladies in 2010. That achievement was evidently just a warm-up for his virtuoso performance in The Gene: An Intimate History, in which he braids science, history, and memoir into an epic with all the range and biblical thunder of Paradise Lost” (The New York Times). In this biography Mukherjee brings to life the quest to understand human heredity and its surprising influence on our lives, personalities, identities, fates, and choices. “Mukherjee expresses abstract intellectual ideas through emotional stories…[and] swaddles his medical rigor with rhapsodic tenderness, surprising vulnerability, and occasional flashes of pure poetry” (The Washington Post). Throughout, the story of Mukherjee’s own family—with its tragic and bewildering history of mental illness—reminds us of the...",1.000000
132340,Gene Hunter: The Story of Neuropsychologist Nancy Wexler (Women's Adventures in Science (Joseph Henry Press)),"Siddhartha Mukherjee onderzoekt aan de hand van zijn eigen familiegeschiedenis - een verleden vol geestesziekte en psychische aandoeningen - de menselijke erfelijkheid en het effect ervan op onze levens, persoonlijkheden, keuzes en lotsbestemmingen. In weergaloos proza beschrijft hij het eeuwenlange onderzoek naar de erfelijkheidskwestie - van Aristoteles en Pythagoras via Mendel en Darwin tot aan de revolutionaire eenentwintigste- eeuwse vernieuwers die het menselijk genoom in kaart brengen. In 'Het gen. Een intieme geschiedenis' verweeft Mukherjee wetenschap en sociale historie met een persoonlijk verhaal, om een onthullende en magistrale geschiedenis te schrijven waarin een wetenschappelijke abstractie tot leven komt. Het boek is onmisbaar voor iedereen die geïnteresseerd is in de morele complexiteit van de huidige wetenschappelijke mogelijkheden om het menselijk genoom te lezen en te schrijven, en voor iedereen die zich bezorgd afvraagt wat de toekomst van de mens behelst.",0.532322
121725,BURN WITCH BURN!,"I AM a medical man specializing in neurology and diseases of the brain. My peculiar field is abnormal psychology, and in it I am recognized as an expert. I am closely connected with two of the foremost hospitals in New York, and have received many honors in this country and abroad. I set this down, risking identification, not through egotism but because I desire to show that I was competent to observe, and competent to bring practiced scientific judgment upon, the singular events I am about to relate...",0.524285
13778,Siddhartha: An Indian Tale (Penguin Twentieth-Century Classics),"A bold translation of Nobel Prize-winner Herman Hesse's most inspirational and beloved work, which was nominated as one of America’s best-loved novels by PBS’s The Great American Read A Penguin Classic Hesse's famous and influential novel, Siddartha, is perhaps the most important and compelling moral allegory our troubled century has produced. Integrating Eastern and Western spiritual traditions with psychoanalysis and philosophy, this strangely simple tale, written with a deep and moving empathy for humanity, has touched the lives of millions since its original publication in 1922. Set in India, Sid

#### Function to find the most similar Books (Recommendation)

In [14]:
# ============================================================
# CELL 15 - Load Metadata and Embeddings for Similarity Search (FUNCTION)
# ============================================================

import numpy as np
import pandas as pd

# modify the column width
pd.set_option('display.max_colwidth', None)

# Load in < 1 second
metadata = pd.read_parquet("../Data/Books_data/books_metadata_only.parquet")
vectors = np.load("../Data/Books_data/books_embeddings_matrix.npy")

print(f"Metadata shape: {metadata.shape}")
print(f"Embeddings shape: {vectors.shape}")

Metadata shape: (209452, 9)
Embeddings shape: (209452, 384)


In [15]:
# ============================================================
# CELL 16 - Save the embedding for all books (FUNCTION)
# ============================================================

# What vstack does is it takes a list of 1D arrays (embeddings) and stacks them vertically to create a 2D array where each row corresponds to a book's embedding. This is the format we need for similarity calculations.
embeddings_books = np.vstack(vectors)
embeddings_books.shape

(209452, 384)

In [16]:
# ============================================================
# CELL 17 - Create a get_similar_books function with the inputs: embeddings, book_index, book_details, top_n
# ============================================================

def get_similar_books(embeddings, book_index, book_details, top_n=5):

    # create book embedding for book_index
    m_embedding = np.array(embeddings[book_index]).reshape(1, -1)
    
    # calculate similarity scores
    similarity_scores = cosine_similarity(m_embedding, embeddings)
    similarity_scores_series = pd.Series(similarity_scores.flatten(), name='similarity_score')
    
    # bring in book info
    books_similarity_scores_df = pd.concat([book_details, similarity_scores_series], axis=1)

    # display book recommendations
    return books_similarity_scores_df.sort_values('similarity_score', ascending=False).iloc[0:top_n+1]

In [17]:
metadata[metadata['Title'].str.contains('universe', case=False)].head(3)

,Title,description,authors_parsed,categories_parsed,year,image,infoLink,ratingsCount,content_for_embedding
175,"Life, the Universe and Everything","Now celebrating the 42nd anniversary of The Hitchhiker’s Guide to the Galaxy, soon to be a Hulu original series! “Wild satire . . . The feckless protagonist, Arthur Dent, is reminiscent of Vonnegut heroes.”—Chicago Tribune The unhappy inhabitants of planet Krikkit are sick of looking at the night sky above their heads—so they plan to destroy it. The universe, that is. Now only five individuals stand between the killer robots of Krikkit and their goal of total annihilation. They are Arthur Dent, a mild-mannered space and time traveler who tries to learn how to fly by throwing himself at the ground and missing; Ford Prefect, his best friend, who decides to go insane to see if he likes it; Slartibartfast, the indomitable vice president of the Campaign for Real Time, who travels in a ship powered by irrational behavior; Zaphod Beeblebrox, the two-headed, three-armed ex-president of the galaxy; and Trillian, the sexy space cadet who is torn between a persistent Thunder God and a very depressed Beeblebrox. How will it all end? Will it end? Only this stalwart crew knows as they try to avert “universal” Armageddon and save life as we know it—and don’t know it! “Adams is one of those rare treasures: an author who, one senses, has as much fun writing as one has reading.”—Arizona Daily Star",[Douglas Adams],[Fiction],2008.0,http://books.google.com/books/content?id=KcWGokt5fsQC&printsec=frontcover&img=1&zoom=1&edge=curl&source=gbs_api,https://play.google.com/store/books/details?id=KcWGokt5fsQC&source=gbs_api,1848.0,"life, the universe and everything douglas adams fiction now celebrating the 42nd anniversary of the hitchhiker’s guide to the galaxy, soon to be a hulu original series! “wild satire . . . the feckless protagonist, arthur dent, is reminiscent of vonnegut heroes.”—chicago tribune the unhappy inhabitants of planet krikkit are sick of looking at the night sky above their heads—so they plan to destroy it. the universe, that is. now only five individuals stand between the killer robots of krikkit and their goal of total annihilation. they are arthur dent, a mild-mannered space and time traveler who tries to learn how to fly by throwing himself at the ground and missing; ford prefect, his best friend, who decides to go insane to see if he likes it; slartibartfast, the indomitable vice president of the campaign for real time, who travels in a ship powered by irrational behavior; zaphod beeblebrox, the two-headed, three-armed ex-president of the galaxy; and trillian, the sexy space cadet who is torn between a persistent thunder god and a very depressed beeblebrox. how will it all end? will it end? only this stalwart crew knows as they try to avert “universal” armageddon and save life as we know it—and don’t know it! “adams is one of those rare treasures: an author who, one senses, has as much fun writing as one has reading.”—arizona daily star"
778,After the Clockwork Universe,"New York Times bestselling author Edward Dolnick brings to light the true story of one of the most pivotal moments in modern intellectual history—when a group of strange, tormented geniuses invented science as we know it, and remade our understanding of the world. Dolnick’s earth-changing story of Isaac Newton, the Royal Society, and the birth of modern science is at once an entertaining romp through the annals of academic history, in the vein of Bill Bryson’s A Short History of Nearly Everything, and a captivating exploration of a defining time for scientific progress, in the tradition of Richard Holmes’ The Age of Wonder.",[Edward Dolnick],[Science],2011.0,http://books.google.com/books/content?id=IJ-GgyfvKL8C&printsec=frontcover&img=1&zoom=1&edge=curl&source=gbs_api,https://play.google.com/store/books/details?id=IJ-GgyfvKL8C&source=gbs_api,95.0,"after the clockwork universe edward dolnick science new york times bes

In [18]:
# ============================================================
# CELL 18 - Provide book recommendations based on a selected book using the get_similar_books function
# ============================================================

get_similar_books(vectors, 175, metadata[['Title', 'description']])

,Title,description,similarity_score
175,"Life, the Universe and Everything","Now celebrating the 42nd anniversary of The Hitchhiker’s Guide to the Galaxy, soon to be a Hulu original series! “Wild satire . . . The feckless protagonist, Arthur Dent, is reminiscent of Vonnegut heroes.”—Chicago Tribune The unhappy inhabitants of planet Krikkit are sick of looking at the night sky above their heads—so they plan to destroy it. The universe, that is. Now only five individuals stand between the killer robots of Krikkit and their goal of total annihilation. They are Arthur Dent, a mild-mannered space and time traveler who tries to learn how to fly by throwing himself at the ground and missing; Ford Prefect, his best friend, who decides to go insane to see if he likes it; Slartibartfast, the indomitable vice president of the Campaign for Real Time, who travels in a ship powered by irrational behavior; Zaphod Beeblebrox, the two-headed, three-armed ex-president of the galaxy; and Trillian, the sexy space cadet who is torn between a persistent Thunder God and a very depressed Beeblebrox. How will it all end? Will it end? Only this stalwart crew knows as they try to avert “universal” Armageddon and save life as we know it—and don’t know it! “Adams is one of those rare treasures: an author who, one senses, has as much fun writing as one has reading.”—Arizona Daily Star",1.000000
207727,Hitchhiker's Guide to the Galaxy : The Authorized Collection (Comic Book),"This is a story about two friends who've known each other for a while. At least Arthur Dent thinks so until Ford Prefect reaveals that he's actually an alien and that the world is about to end! Not long after their conversation, a Vogon spaceship appears and announces that earth will be destroyed shortly because it is in the way of a new intergalactic highway bypass that is being built through our galaxy. Ford and Arthur manage to board the ship before the earth is demolished, and they are quickly discovered - ejected into space by the Vogon captain! Douglas Adams' wildly funny, wickedly clever sci-fi novel is collected here in a three-installment, comic ebook series. So grab your towel, stick a babel fish in your ear, and get set to join Arthur Dent, Ford Prefect, Zaphod Beeblebrox, Trillian and Marvin the Paranoid Android on the ultimate adventure of several lifetimes.",0.579361
111875,ESPN Sports Almanac 2006,"An Englishman's continuing search through space and time for a decent cup of tea . . . Arthur Dent's accidental association with that wholly remarkable book, The Hitchhiker's Guide to the Galaxy, has not been entirely without incident. Arthur has traveled the length, breadth, and depth of known, and unknown, space. He has stumbled forward and backward through time. He has been blown up, reassembled, cruelly imprisoned, horribly released, and colorfully insulted more than is strictly necessary. And of course Arthur Dent has comprehensively failed to grasp the meaning of life, the universe, and everything. Arthur has finally made it home to Earth, but that does not mean he has escaped his fate. Arthur's chances of getting his hands on a decent cuppa have evaporated rapidly, along with all the world's oceans. For no sooner has he touched down on the planet Earth than he finds out that it is about to be blown up . . . again. And Another Thing . . . is the rather unexpected, but very welcome, sixth installment of the Hitchhiker's Guide to the Galaxy series. It features a pantheon of unemployed gods, everyone's favorite renegade Galactic President, a lovestruck green alien, an irritating computer, and at least one very large slab of cheese.",0.577475
7766,Vacuum Diagrams: Stories of the Xeelee Sequence,"""And everywhere the Humans went, they found life ..."" This dazzling future history, winner of the 2000 Philip K. Dick Award, is the most ambitious and exciting since Asimov's classic Foundation saga. It tells the story of Humankind -- all the way to the end of the Universe itself. 